# CURB - Chronic-Offender Intelligence
### notebook 03 - Problem Statement 1 (Parking-Induced Congestion)

Two data-grounded views that add a layer no other team is likely to have:

1. **Repeat-offender vehicles** - anonymized vehicle IDs cited 2+ times: how often,
   where, and whether they're *Habitual* (same spot) or *Roaming* (many spots).
2. **Recidivist locations** - spots where the **same** vehicles keep returning. A spot
   with 500 citations from 500 different cars is transient churn; 500 from 50 returning
   cars is a habitual-parking problem that needs a structural fix, not just a patrol.

**Honesty (put on the methodology slide):** these are repeat *detections*, not proven
repeat *offences* - a car caught often may simply park where enforcement is heavy. Treat
as candidates for review, not a verdict. Vehicle IDs are anonymized (aggregate targeting,
never owner identification). Everything traces to the provided dataset only.

In [ ]:
import os, sys, json
from pathlib import Path
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config.py").exists() and (REPO_ROOT.parent / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT)); os.chdir(REPO_ROOT)
import config, pandas as pd, matplotlib.pyplot as plt
from src.offenders import run
DATA_PATH = config.DATA_PATH
pd.set_option("display.max_colwidth", 34)
print("data:", DATA_PATH, "exists:", os.path.exists(DATA_PATH))

## 1 - Run the analysis

In [ ]:
veh, loc = run(DATA_PATH)
print(f"{len(veh):,} repeat-offender vehicles  |  {len(loc):,} locations assessed for recidivism")

## 2 - Repeat-offender vehicles

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
veh["pattern"].value_counts().plot.bar(ax=ax[0], color=["#7C8DA6","#E5564B","#4C9BE6"])
ax[0].set_title("Offender pattern"); ax[0].set_ylabel("vehicles"); ax[0].tick_params(axis="x", rotation=15)
veh["violations"].plot.hist(bins=range(2, 50), ax=ax[1], color="#F0A93B")
ax[1].set_title("Citations per repeat vehicle"); ax[1].set_xlabel("citations")
plt.tight_layout(); plt.show()

veh.head(12)[["violations","distinct_locations","top_location","top_location_share",
              "span_days","vehicle_type","pattern"]]

The *Habitual (one spot)* vehicles are the most actionable: a plate cited many times
with a high `top_location_share` is parking illegally in the same place repeatedly - a
clean candidate for a targeted notice.

In [ ]:
hab = veh[veh.pattern == "Habitual (one spot)"].head(10)
hab[["violations","top_location","top_location_share","vehicle_type","span_days"]]

## 3 - Recidivist locations (same vehicles returning)

In [ ]:
print(loc.recidivism.value_counts().to_string())
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
loc.recidivism.value_counts().reindex(["Habitual","Mixed","Transient"]).plot.bar(
    ax=ax[0], color=["#E5564B","#F0A93B","#7C8DA6"])
ax[0].set_title("Location recidivism"); ax[0].set_ylabel("locations")
col = loc.recidivism.map({"Habitual":"#E5564B","Mixed":"#F0A93B","Transient":"#7C8DA6"})
ax[1].scatter(loc.violations, loc.repeat_share, s=14, c=col, alpha=.55)
ax[1].set_xlabel("citations at the spot"); ax[1].set_ylabel("share from returning vehicles")
ax[1].set_title("Volume vs recidivism"); ax[1].set_xlim(0, 200)
plt.tight_layout(); plt.show()

In [ ]:
loc.head(12)[["rank","name","violations","distinct_vehicles","repeat_vehicles",
              "repeat_share","recidivism","impact_score"]]

A **Habitual** location (high `repeat_share`, few distinct vehicles) argues for a
*structural* response - designated/permit parking - because the same vehicles return
daily. A **Transient** spot (many different vehicles) argues for *enforcement presence*.
This sharpens the root-cause prescription from notebook 02.

## 4 - Files written (for the deck / optional frontend use)

In [ ]:
for f in ["outputs/curb_offenders.json","outputs/curb_offender_vehicles.csv","outputs/curb_recidivist_locations.csv"]:
    print("wrote", f, "-", os.path.getsize(f), "bytes")
j = json.load(open("outputs/curb_offenders.json"))
print("\nmeta:", json.dumps(j["meta"], indent=2))
print("\nsample recidivist location:"); print(json.dumps(j["recidivist_locations"][0], indent=2))

## Methodology note
- Repeat **detections**, not proven offences (enforcement-location bias acknowledged).
- Anonymized vehicle IDs -> aggregate targeting only, never owner identification.
- A location is assessed for recidivism only if it has >= 20 citations (below that, a
  "100% repeat" is just noise). Thresholds are documented constants in `src/offenders.py`.
- Provided dataset only; no external data.